# Increase Resolution — distributed (tile-level), roles in action

This notebook runs the distributed pipeline the way a real deployment would: a **coordinator** machine that
installs only **`increase-resolution[cpu]`** (no GPU) and **worker** machines that install only
**`increase-resolution[gpu]`**. Tiles are passed between them.

| Step | Role | Env in this notebook | Install |
|---|---|---|---|
| `split(image)` | **coordinator** | `/opt/coordinator` | `increase-resolution[cpu]` |
| `TileWorker.infer(tile)` | **worker** | `/opt/worker` | `increase-resolution[gpu]` |
| `merge(tiles, layout)` | **coordinator** | `/opt/coordinator` | `increase-resolution[cpu]` |

We build **two separate virtualenvs** so you can see the split for real: the coordinator env has **no torch /
no CUDA**, the worker env has the GPU stack. The tiles cross between them as plain `.npy` files (stand-in for
your queue / RPC / Ray transport). The result is identical to the single-call `execute`.

**Prerequisite:** `BRIA_API_TOKEN`, and (for the worker) the A10 / CUDA 11.7.1 / TensorRT 8.4.1.5 runtime from
the README (this notebook's kernel already has the `LD_PRELOAD`/env set).

## 1. CodeArtifact token
One token; both roles install with it (username `aws`).

In [ ]:
import os, subprocess, sys, json, glob
import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN before running this notebook.")
resp = requests.get(
    "https://engine.prod.bria-api.com/v2/auth/access/code_artifact",
    params={"repository": "bria-increase-res"}, headers={"api_token": BRIA_API_TOKEN}, timeout=60,
)
resp.raise_for_status()
from urllib.parse import quote
pw = resp.json()["result"]["authorization_token"].strip()
CA = "https://aws:" + quote(pw, safe="") + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-increase-res/simple/"
print("CodeArtifact credential acquired.")

## 2a. Coordinator install — `increase-resolution[cpu]`

The coordinator does `split`/`merge` only. It needs **no GPU, no torch, no TensorRT** — just numpy/opencv/
pillow. We build it in its own venv:

```bash
pip install "increase-resolution[cpu]" --extra-index-url "$CA"
```

In [ ]:
subprocess.check_call([sys.executable, "-m", "venv", "/opt/coordinator"])
subprocess.check_call(["/opt/coordinator/bin/pip", "install", "-q", "--upgrade",
                       "increase-resolution[cpu]", "--extra-index-url", CA])
print("coordinator env ready:")
subprocess.check_call(["/opt/coordinator/bin/pip", "list"])  # note: no torch / no tensorrt below

**Proof the coordinator is GPU-free** — importing torch in it must fail:

In [ ]:
r = subprocess.run(["/opt/coordinator/bin/python", "-c", "import torch"],
                   capture_output=True, text=True)
print("exit code:", r.returncode)
print(r.stderr.strip().splitlines()[-1] if r.stderr else "(torch imported?!)")
assert r.returncode != 0, "coordinator should NOT have torch"
print("\n=> coordinator[cpu] has no torch/CUDA — pure split/merge, runs on any CPU box.")

## 2b. Worker install — `increase-resolution[gpu]`

The worker does **only** tile inference (`TileWorker.infer`): torch (cu117) + nvidia-tensorrt. Its own venv:

```bash
pip install "increase-resolution[gpu]" \
  --extra-index-url https://download.pytorch.org/whl/cu117 \
  --extra-index-url https://pypi.ngc.nvidia.com --extra-index-url "$CA"
```

In [ ]:
subprocess.check_call([sys.executable, "-m", "venv", "/opt/worker"])
subprocess.check_call(["/opt/worker/bin/pip", "install", "-q", "--upgrade", "increase-resolution[gpu]",
                       "--extra-index-url", "https://download.pytorch.org/whl/cu117",
                       "--extra-index-url", "https://pypi.ngc.nvidia.com",
                       "--extra-index-url", CA])
r = subprocess.run(["/opt/worker/bin/python", "-c",
                    "import torch, tensorrt; print('worker env: torch', torch.__version__, '| tensorrt', tensorrt.__version__)"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

## 3. Engine + sample — worker needs the engine, coordinator needs the image

Only the worker needs the `.engine`. Replace the URLs with the links Bria gave you (already-cached files are
reused).

In [ ]:
from pathlib import Path
cache = Path(os.path.expanduser("~/.cache/bria/increase-resolution/")); cache.mkdir(parents=True, exist_ok=True)
ENGINE_URLS = {2: "<BRIA_PROVIDED_URL_FOR_increase_resolution2.engine>",
               4: "<BRIA_PROVIDED_URL_FOR_increase_resolution4.engine>"}
ENGINE_PATHS = {}
for scale, url in ENGINE_URLS.items():
    dest = cache / f"increase_resolution{scale}.engine"
    if not dest.exists():
        tmp = dest.with_suffix(".engine.part")
        with requests.get(url, stream=True, timeout=600) as rr:
            rr.raise_for_status()
            with open(tmp, "wb") as f:
                for chunk in rr.iter_content(1 << 20): f.write(chunk)
        tmp.rename(dest)
    ENGINE_PATHS[scale] = str(dest); print(f"x{scale} engine:", dest)

SCALE = 4
WORK = "/tmp/dist_demo"; os.makedirs(WORK, exist_ok=True)
IMG = f"{WORK}/input.png"
open(IMG, "wb").write(requests.get("https://bria-test-images.s3.us-east-1.amazonaws.com/starry_2000x2000.png").content)
print("sample image ->", IMG)

## 4. Split — on the coordinator (`[cpu]`)

Run `split` **in the coordinator env**. It writes each tile as a `.npy` (what you'd ship to workers) plus a
`layout.json`. No engine, no GPU.

In [ ]:
%%writefile coordinator_split.py
import sys, os, json
import numpy as np
from PIL import Image
from increase_resolution import split   # only needs [cpu]

img_path, out_dir, scale = sys.argv[1], sys.argv[2], int(sys.argv[3])
os.makedirs(f"{out_dir}/tiles", exist_ok=True)
res = split(Image.open(img_path), scale=scale)
for i, t in enumerate(res.tiles):
    np.save(f"{out_dir}/tiles/{i:04d}.npy", t)
json.dump(res.layout, open(f"{out_dir}/layout.json", "w"))
if res.alpha is not None:
    np.save(f"{out_dir}/alpha.npy", res.alpha)
print(f"[coordinator/cpu] split into {len(res.tiles)} tiles of {res.tiles[0].shape} -> {out_dir}/tiles/")

In [ ]:
subprocess.check_call(["/opt/coordinator/bin/python", "coordinator_split.py", IMG, WORK, str(SCALE)])
print("tiles on disk:", len(glob.glob(f"{WORK}/tiles/*.npy")))

## 5. Tile inference — on the worker (`[gpu]`)

Run `TileWorker.infer` **in the worker env**, reading the tiles the coordinator produced and writing upscaled
tiles back. This is the only GPU step. In production this runs on each worker machine, fanned out.

In [ ]:
%%writefile worker_infer.py
import sys, os, glob
import numpy as np
from increase_resolution import TileWorker   # needs [gpu]

tiles_dir, up_dir, engine = sys.argv[1], sys.argv[2], sys.argv[3]
os.makedirs(up_dir, exist_ok=True)
w = TileWorker(engine).setup()               # each worker builds this once
files = sorted(glob.glob(f"{tiles_dir}/*.npy"))
for f in files:
    np.save(f"{up_dir}/{os.path.basename(f)}", w.infer(np.load(f)))
w.close()
print(f"[worker/gpu] upscaled {len(files)} tiles -> {up_dir}/")

In [ ]:
subprocess.check_call(["/opt/worker/bin/python", "worker_infer.py",
                       f"{WORK}/tiles", f"{WORK}/up", ENGINE_PATHS[SCALE]])
print("upscaled tiles on disk:", len(glob.glob(f"{WORK}/up/*.npy")))

## 6. Merge — back on the coordinator (`[cpu]`)
Stitch the upscaled tiles into the final image — again no GPU.

In [ ]:
%%writefile coordinator_merge.py
import sys, os, glob, json
import numpy as np
from increase_resolution import merge   # only needs [cpu]

up_dir, layout_path, alpha_path, out_png = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4]
layout = json.load(open(layout_path))
upscaled = [np.load(f) for f in sorted(glob.glob(f"{up_dir}/*.npy"))]
alpha = np.load(alpha_path) if os.path.exists(alpha_path) else None
out = merge(upscaled, layout, alpha)
out.save(out_png)
print(f"[coordinator/cpu] merged {len(upscaled)} tiles -> {out_png} {out.size}")

In [ ]:
subprocess.check_call(["/opt/coordinator/bin/python", "coordinator_merge.py",
                       f"{WORK}/up", f"{WORK}/layout.json", f"{WORK}/alpha.npy", f"{WORK}/output_distributed.png"])
from PIL import Image
from IPython.display import display
preview = Image.open(f"{WORK}/output_distributed.png"); preview.thumbnail((900, 900))
display(preview)

## 7. Correctness — distributed == single-call

The coordinator+worker result must equal the single-call `IncreaseResolution.execute` (which needs `[all]`).
This kernel has `[all]`, so we run the reference here and compare.

In [ ]:
import numpy as np
from PIL import Image
from increase_resolution import IncreaseResolution, IncreaseResolutionConfig, IncreaseResolutionInput

pipe = IncreaseResolution(config=IncreaseResolutionConfig(engine_paths={int(k): v for k, v in ENGINE_PATHS.items()}))
pipe.setup()
ref = pipe.execute(IncreaseResolutionInput(image=Image.open(IMG), desired_increase=SCALE)).image
pipe.cleanup()
dist = Image.open(f"{WORK}/output_distributed.png")
mae = np.abs(np.asarray(dist).astype(int) - np.asarray(ref).astype(int)).mean()
print(f"distributed([cpu]+[gpu]) vs single-call[all] MAE = {mae:.4f}/255  ({'MATCH' if mae < 1.0 else 'MISMATCH'})")